# SEC Insider Transactions Data (2025)

This downloaded SEC Insider Transactions data sets for the year 2025 are stored in the `data` folder.
Data source: https://www.sec.gov/data-research/sec-markets-data/insider-transactions-data-sets

In [ ]:
import os
import requests
import zipfile
import io
import csv

# Define the range of years and quarters to download
start_year = 2025
start_quarter = 1
end_year = 2025
end_quarter = 4

# Set SEC-compliant User-Agent
# Note: The SEC requires the User-Agent to conform to the format:
# 'Sample Company Name AdminContact@<sample company domain>.com'
headers = {
    'User-Agent': 'info@example.com',
    'Accept-Encoding': 'gzip, deflate',
    'Host': 'www.sec.gov'
}
base_url = 'https://www.sec.gov/files/structureddata/data/insider-transactions-data-sets/'

# Create a list to store the data directories for each quarter
data_dirs = []
quarters = []

# Loop through the specified years and quarters to download, extract, and convert the data
for year in range(start_year, end_year + 1):
    for quarter in range(start_quarter, end_quarter + 1):
        quarters.append(f"{year}q{quarter}")
        quarter_str = f'q{quarter:1d}'
        data_dir = f'../data/sec/{year}{quarter_str}'
        os.makedirs(data_dir, exist_ok=True)
        data_dirs.append(data_dir)
        filename = f"{year}{quarter_str}_form345.zip"
        url = f"{base_url}{filename}"
    
        print(f"Fetching {filename}...")
        response = requests.get(url, headers=headers)
    
        if response.status_code == 200:
            # Save the zip file
            zip_path = os.path.join(data_dir, filename)
            with open(zip_path, 'wb') as f:
                f.write(response.content)
            print(f"Successfully downloaded {filename}")
        
            # Extract the zip file into its own quarter directory
            print(f"Extracting {filename} into {data_dir}...")
            # extract_dir = os.path.join(data_dir, quarter)
            os.makedirs(data_dir, exist_ok=True)
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print(f"Extraction complete for {quarter}.\n")
            
            # Remove the zip file
            os.remove(os.path.join(data_dir, filename))
            
            # Convert TSV files to CSV
            for filename in os.listdir(data_dir):
                if filename.endswith('.tsv'):
                    tsv_path = os.path.join(data_dir, filename)
                    csv_path = os.path.splitext(tsv_path)[0] + '.csv'

                    # Open the TSV file and create a CSV writer
                    with open(tsv_path, 'r') as tsv_file, open(csv_path, 'w', newline='') as csv_file:
                        tsv_reader = csv.reader(tsv_file, delimiter='\t')
                        csv_writer = csv.writer(csv_file)

                        # Write each row from the TSV file to the CSV file
                        for row in tsv_reader:
                            csv_writer.writerow(row)

                    print(f"Conversion complete for {filename}.\n")

                    # Remove the original TSV file
                    os.remove(tsv_path)
        elif response.status_code == 404:
            print(f"File {filename} not found (Status code: 404). It may not be released yet.\n")
        elif response.status_code == 403:
            print(f"Access forbidden for {filename} (Status code: 403). Please verify your User-Agent.\n")
        else:
            print(f"Failed to download {filename} (Status code: {response.status_code})\n")

    print("All requested downloads processed.")                    

# os.makedirs(data_dir, exist_ok=True)

# List of 2025 quarters

#for year in range(2025, 2026):
#    for quarter in range(1, 5):
#        quarters.append(f"{year}q{quarter}")
# quarters = ['2025q1', '2025q2', '2025q3', '2025q4']

# for quarter in quarters:
#    data_dir = f'../data/sec/{year}{quarter_str}'

Fetching 2025q1_form345.zip...
Successfully downloaded 2025q1_form345.zip
Extracting 2025q1_form345.zip into ../data/sec/2025q1...
Extraction complete for 1.

Conversion complete for SUBMISSION.tsv.

Conversion complete for DERIV_HOLDING.tsv.

Conversion complete for NONDERIV_TRANS.tsv.

Conversion complete for FOOTNOTES.tsv.

Conversion complete for OWNER_SIGNATURE.tsv.

Conversion complete for REPORTINGOWNER.tsv.

Conversion complete for NONDERIV_HOLDING.tsv.

Conversion complete for DERIV_TRANS.tsv.

Fetching 2025q2_form345.zip...
Successfully downloaded 2025q2_form345.zip
Extracting 2025q2_form345.zip into ../data/sec/2025q2...
Extraction complete for 2.

Conversion complete for SUBMISSION.tsv.

Conversion complete for DERIV_HOLDING.tsv.

Conversion complete for NONDERIV_TRANS.tsv.

Conversion complete for FOOTNOTES.tsv.

Conversion complete for OWNER_SIGNATURE.tsv.

Conversion complete for REPORTINGOWNER.tsv.

Conversion complete for NONDERIV_HOLDING.tsv.

Conversion complete for 

In [ ]:
import pandas as pd

# File paths for 2025 Q4 data
data_dir_q4 = '../data/sec/2025q4'

# 1. Load Submissions (contains Company Names and Tickers)
# low_memory=False used to handle mixed data types in TSV
sub_file = f'{data_dir_q4}/SUBMISSION.tsv'
df_sub = pd.read_csv(sub_file, sep='\t', low_memory=False)

print(f"Submissions shape: {df_sub.shape}")

df_sub

In [ ]:
# 2. Load Non-derivative Transactions (the actual trades)
trans_file = f'{data_dir_q4}/NONDERIV_TRANS.tsv'
df_trans = pd.read_csv(trans_file, sep='\t', low_memory=False)
print(f"Transactions shape: {df_trans.shape}")

df_trans

In [ ]:
# 3. Merge to link transactions with company details
df_merged = pd.merge(
    df_trans, 
    df_sub[['ACCESSION_NUMBER', 'ISSUERNAME', 'ISSUERTRADINGSYMBOL']], 
    on='ACCESSION_NUMBER', 
    how='inner'
)

df_merged



In [ ]:
# Convert shares and prices to numeric, handling any parsing errors
df_merged['TRANS_SHARES'] = pd.to_numeric(df_merged['TRANS_SHARES'], errors='coerce').fillna(0)
df_merged['TRANS_PRICEPERSHARE'] = pd.to_numeric(df_merged['TRANS_PRICEPERSHARE'], errors='coerce').fillna(0)

# 4. Filter for only basic Acquired (A) or Disposed (D) trades
df_ad = df_merged[df_merged['TRANS_ACQUIRED_DISP_CD'].isin(['A', 'D'])].copy()

df_ad


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 5. Plot the raw counts of Buys vs Sells
print("\n--- Transaction Counts ---")
print(df_ad['TRANS_ACQUIRED_DISP_CD'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df_ad, x='TRANS_ACQUIRED_DISP_CD', palette='viridis')
plt.title('Count of Transactions: Acquired (A) vs Disposed (D)')
plt.show()

# 6. Calculate total monetary value of transactions
df_ad['TRANS_VALUE'] = df_ad['TRANS_SHARES'] * df_ad['TRANS_PRICEPERSHARE']

# Group by Issuer and Acquired/Disposed code to find the top companies
top_companies = df_ad.groupby(['ISSUERNAME', 'TRANS_ACQUIRED_DISP_CD'])['TRANS_VALUE'].sum().unstack(fill_value=0)

print("\n--- Top 10 Companies by Acquired (Buy) Value ---")
top_acquired = top_companies.sort_values(by='A', ascending=False).head(10)['A']
print(top_acquired.map('${:,.2f}'.format))

print("\n--- Top 10 Companies by Disposed (Sell) Value ---")
top_disposed = top_companies.sort_values(by='D', ascending=False).head(10)['D']
print(top_disposed.map('${:,.2f}'.format))
